# CRAFT Phase 1: SFT Warmup

Trains a QLoRA adapter on Phi-3-Mini using GSM8K, AQuA-RAT, StrategyQA, and MMLU.

**How this notebook works:**
1. Gets the latest code from GitHub
2. ALWAYS copies checkpoints from the Kaggle dataset (they are never in git)
3. Resumes training from the latest checkpoint

**Why checkpoints are never in git:** Model checkpoint folders are hundreds of MB each. They are stored as a Kaggle dataset and mounted at `/kaggle/input/`. Every session, we copy them into `/kaggle/working/` where training can resume from them.

In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers==4.44.0 trl==0.10.1 peft==0.12.0 \
              accelerate==0.34.2 datasets==2.20.0 bitsandbytes \
              loguru pyyaml scipy numpy rich
print("✅ Dependencies installed")

In [ ]:
# ── CELL 2: Clone or update the GitHub repo ──────────────────────────────────
# This gives us the latest code (train_sft.py, configs, src/ modules)
# NOTE: Checkpoints are NOT in git. They come from the Kaggle dataset in Cell 3.

from kaggle_secrets import UserSecretsClient
import os
import subprocess

user_secrets = UserSecretsClient()
git_token = user_secrets.get_secret("GITHUB_PAT")
username = "VishalGastu30"
repo_name = "CRAFT-SLM-Reasoning"
working_dir = f"/kaggle/working/{repo_name}"

if os.path.exists(working_dir):
    print(f"📁 Repository already exists at {working_dir}")
    print("   Pulling latest code from GitHub...")
    result = subprocess.run(
        ["git", "pull", f"https://{git_token}@github.com/{username}/{repo_name}.git"],
        cwd=working_dir,
        capture_output=True, text=True
    )
    print(result.stdout.strip() or "Already up to date.")
    if result.returncode != 0:
        print(f"⚠️  git pull warning: {result.stderr.strip()}")
        print("   Continuing with existing code.")
else:
    print(f"📦 Cloning repository to {working_dir}...")
    !git clone https://{git_token}@github.com/{username}/{repo_name}.git {working_dir}
    print("✅ Clone complete.")

print(f"\n✅ Code is ready at: {working_dir}")

In [ ]:
# ── CELL 3: Copy checkpoints from dataset → working directory ─────────────────
#
# THIS IS THE CRITICAL CELL. Read this comment carefully before changing anything.
#
# PROBLEM THIS SOLVES:
#   - Checkpoints are saved to /kaggle/working/ during training
#   - Between Kaggle sessions, /kaggle/working/ is WIPED
#   - To preserve checkpoints, they were saved as a Kaggle dataset
#   - The dataset is mounted at /kaggle/input/ (read-only)
#   - Training needs checkpoints in /kaggle/working/ (read-write)
#   - So: every session, we MUST copy checkpoints from input → working
#
# WHY THE OLD CODE FAILED:
#   - The old Cell 4 used shutil.copytree for the ENTIRE repo
#   - It checked "if dst exists, skip the copy" → skipped when git clone already ran
#   - This means checkpoints were NEVER copied from the dataset
#   - The git repo has no checkpoints (they are .gitignored)
#   - Result: "No checkpoints found" → sys.exit(1)
#
# THIS FIX:
#   - Separates code (git) from checkpoints (dataset)
#   - ALWAYS copies checkpoints from dataset regardless of repo state
#   - Never depends on the git repo having checkpoints

import os
import shutil
import sys

# Paths
DATASET_ROOT = "/kaggle/input/datasets/vishalgastu30/ver-1-phase-1/CRAFT-SLM-Reasoning"
WORKING_DIR  = "/kaggle/working/CRAFT-SLM-Reasoning"

DATASET_SFT_CKPT  = os.path.join(DATASET_ROOT, "checkpoints", "sft")
WORKING_SFT_CKPT  = os.path.join(WORKING_DIR,  "checkpoints", "sft")

# ── Step 1: Move into the working repo ────────────────────────────────────────
if not os.path.exists(WORKING_DIR):
    print(f"❌ Working directory missing: {WORKING_DIR}")
    print("   Run Cell 2 (git clone) first, then rerun this cell.")
    sys.exit(1)

os.chdir(WORKING_DIR)
print(f"📁 Working directory: {os.getcwd()}")

# ── Step 2: Check what's in the dataset ───────────────────────────────────────
print(f"\n🔍 Looking for checkpoints in dataset at:")
print(f"   {DATASET_SFT_CKPT}")

dataset_has_checkpoints = os.path.exists(DATASET_SFT_CKPT)

if dataset_has_checkpoints:
    dataset_ckpts = sorted(
        [d for d in os.listdir(DATASET_SFT_CKPT) if d.startswith("checkpoint-")]
    )
    print(f"✅ Found {len(dataset_ckpts)} checkpoint(s) in dataset:")
    for ckpt in dataset_ckpts:
        ckpt_size = sum(
            os.path.getsize(os.path.join(DATASET_SFT_CKPT, ckpt, f))
            for f in os.listdir(os.path.join(DATASET_SFT_CKPT, ckpt))
            if os.path.isfile(os.path.join(DATASET_SFT_CKPT, ckpt, f))
        ) / 1e6
        print(f"   • {ckpt}  ({ckpt_size:.0f} MB)")
else:
    print("⚠️  No checkpoints found in dataset.")
    print("   This means training will start from scratch (epoch 1, step 1).")
    print("   This is normal if this is your very first training run.")

# ── Step 3: ALWAYS copy checkpoints from dataset → working ────────────────────
# We do this unconditionally because:
#   a) The working directory is empty after a new Kaggle session
#   b) Even if working dir exists (from git clone), it has no checkpoints

if dataset_has_checkpoints:
    print(f"\n📦 Copying checkpoints from dataset to working directory...")
    print(f"   From: {DATASET_SFT_CKPT}")
    print(f"   To:   {WORKING_SFT_CKPT}")

    # Remove stale working checkpoints if they exist
    # (Could be outdated from a previous interrupted session)
    if os.path.exists(WORKING_SFT_CKPT):
        print(f"   🗑️  Removing existing working checkpoints (replacing with dataset version)...")
        shutil.rmtree(WORKING_SFT_CKPT)

    # Ensure parent directory exists
    os.makedirs(os.path.join(WORKING_DIR, "checkpoints"), exist_ok=True)

    # Copy checkpoints from read-only dataset to read-write working dir
    shutil.copytree(DATASET_SFT_CKPT, WORKING_SFT_CKPT)

    # Verify the copy worked
    copied_ckpts = sorted(
        [d for d in os.listdir(WORKING_SFT_CKPT) if d.startswith("checkpoint-")]
    )
    print(f"✅ Successfully copied {len(copied_ckpts)} checkpoint(s):")
    for ckpt in copied_ckpts:
        print(f"   • {ckpt}")
else:
    # No checkpoints in dataset — create the sft directory so training can start fresh
    os.makedirs(WORKING_SFT_CKPT, exist_ok=True)
    print("\n📁 Created empty checkpoints/sft/ — training will start from scratch.")

# ── Step 4: Also copy RL checkpoints if they exist in the dataset ─────────────
DATASET_RL_CKPT = os.path.join(DATASET_ROOT, "checkpoints", "rl")
WORKING_RL_CKPT = os.path.join(WORKING_DIR,  "checkpoints", "rl")

if os.path.exists(DATASET_RL_CKPT) and not os.path.exists(WORKING_RL_CKPT):
    print(f"\n📦 Also copying RL checkpoints from dataset...")
    shutil.copytree(DATASET_RL_CKPT, WORKING_RL_CKPT)
    rl_ckpts = sorted([d for d in os.listdir(WORKING_RL_CKPT) if d.startswith("checkpoint-")])
    print(f"✅ RL checkpoints copied: {', '.join(rl_ckpts) or 'none found'}")

# ── Step 5: Verify all required files are present ─────────────────────────────
print("\n🔍 Verifying required files...")

required_files = [
    "src/phase1_sft/train_sft.py",
    "src/phase1_sft/data_loader.py",
    "src/phase1_sft/sft_config.py",
    "src/config/phi3_mini.yaml",     
]

all_ok = True
for f in required_files:
    path = os.path.join(WORKING_DIR, f)
    if os.path.exists(path):
        print(f"   ✅ {f}")
    else:
        print(f"   ❌ MISSING: {f}")
        all_ok = False

if not all_ok:
    print("\n❌ Some required files are missing. Check your GitHub repo is up to date.")
    sys.exit(1)

# ── Step 6: Show disk usage ────────────────────────────────────────────────────
print("\n💾 Disk usage:")
os.system("df -h /kaggle/working | tail -1")
os.system("du -sh /kaggle/working/CRAFT-SLM-Reasoning/checkpoints/ 2>/dev/null || echo '   (no checkpoints dir)'")

# ── Final summary ──────────────────────────────────────────────────────────────
print("\n" + "="*60)
if dataset_has_checkpoints:
    print("✅ SETUP COMPLETE — Training will RESUME from latest checkpoint")
    print(f"   Latest checkpoint: {sorted(copied_ckpts)[-1]}")
else:
    print("✅ SETUP COMPLETE — Training will START FROM SCRATCH")
print("   Run the next cell to begin training.")
print("="*60)

In [ ]:
# ── CELL 4: Run SFT training ─────────────────────────────────────────────────
# --resume flag: automatically finds the latest checkpoint and continues from it
# If no checkpoints exist, training starts from scratch (epoch 1, step 1)

!python -m src.phase1_sft.train_sft \
    --config phi3_mini \
    --hardware kaggle \
    --output checkpoints/sft \
    --resume

In [ ]:
# ── CELL 5: Verify training completed ────────────────────────────────────────
import os
import json

WORKING_DIR = "/kaggle/working/CRAFT-SLM-Reasoning"
os.chdir(WORKING_DIR)

final_path = "checkpoints/sft/final"

if os.path.exists(final_path):
    contents = os.listdir(final_path)
    print(f"✅ SFT training COMPLETED. Final weights at: {final_path}")
    print(f"   Files: {contents}")

    # Check the adapter config to confirm it's a valid PEFT adapter
    adapter_cfg = os.path.join(final_path, "adapter_config.json")
    if os.path.exists(adapter_cfg):
        with open(adapter_cfg) as f:
            cfg = json.load(f)
        print(f"   Base model: {cfg.get('base_model_name_or_path', 'unknown')}")
        print(f"   LoRA rank:  {cfg.get('r', 'unknown')}")
    else:
        print("   ⚠️  adapter_config.json not found — checkpoint may be incomplete")
else:
    print("❌ Final checkpoint NOT found at checkpoints/sft/final")
    print("   Training may still be in progress, or it crashed.")
    print("   Check the logs above for the last checkpoint saved.")

# Show all checkpoints that exist
print("\n📁 All SFT checkpoints:")
sft_dir = "checkpoints/sft"
if os.path.exists(sft_dir):
    items = sorted(os.listdir(sft_dir))
    for item in items:
        item_path = os.path.join(sft_dir, item)
        if os.path.isdir(item_path):
            size_mb = sum(
                os.path.getsize(os.path.join(item_path, f))
                for f in os.listdir(item_path)
                if os.path.isfile(os.path.join(item_path, f))
            ) / 1e6
            print(f"   • {item}  ({size_mb:.0f} MB)")
else:
    print("   (checkpoints/sft/ not found)")

print("\n💾 Final disk usage:")
os.system("df -h /kaggle/working | tail -1")

In [ ]:
# ── CELL 6: Save checkpoint to Kaggle output (for next session) ──────────────
#
# IMPORTANT: Kaggle wipes /kaggle/working/ after each session.
# To preserve your checkpoints, copy them to /kaggle/working/Outputs
# which gets saved as a Kaggle dataset version when you "Save Version".
#
# After running this cell:
#   1. Click "Save Version" in the top right
#   2. In the next session's dataset input, use this version
#   3. Cell 3 will copy the checkpoints back into the working directory

import os
import shutil

WORKING_DIR = "/kaggle/working/CRAFT-SLM-Reasoning"
OUTPUT_DIR  = "/kaggle/working/Outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Copy entire checkpoints folder to outputs
ckpt_src = os.path.join(WORKING_DIR, "checkpoints")
ckpt_dst = os.path.join(OUTPUT_DIR, "checkpoints")

if os.path.exists(ckpt_src):
    if os.path.exists(ckpt_dst):
        shutil.rmtree(ckpt_dst)
    shutil.copytree(ckpt_src, ckpt_dst)

    # Report what was saved
    print("✅ Checkpoints copied to Kaggle output (will be saved in next version):")
    for root, dirs, files in os.walk(ckpt_dst):
        level = root.replace(ckpt_dst, '').count(os.sep)
        indent = '  ' * level
        folder = os.path.basename(root)
        if level <= 2:  # Only show top levels
            size_mb = sum(
                os.path.getsize(os.path.join(root, f)) for f in files
            ) / 1e6
            print(f"  {indent}{folder}/  ({size_mb:.0f} MB, {len(files)} files)")
else:
    print("❌ No checkpoints directory found at:", ckpt_src)

print("\n💾 Output directory size:")
os.system(f"du -sh {OUTPUT_DIR}")
print("\n📌 NEXT STEPS:")
print("   1. Click 'Save Version' → 'Save & Run All (Commit)'")
print("   2. In the next Kaggle session, update the dataset version in the input")
print("   3. Cell 3 will automatically copy these checkpoints to the working directory")